# Netflix Content Strategy — Exploratory Analysis

Standalone notebook version of the analysis behind the dashboard. Uses the same `core/etl.py` loaders as the Flask app, so anything you see here is consistent with what's on the live dashboard — this notebook is for deeper ad-hoc digging (e.g. testing a new hypothesis before turning it into a dashboard feature).

**Before running:** activate the project's virtualenv and run this notebook from the project root (`jupyter notebook notebooks/exploratory_analysis.ipynb`), or adjust `sys.path` below.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from core import etl

sns.set_theme(style='darkgrid', palette='deep')
plt.rcParams['figure.figsize'] = (10, 5)

movies = etl.load_movie_market()
engagement = etl.load_engagement()
print(f'movies: {movies.shape}, engagement: {engagement.shape}')

## 1. Movie market — content volume over time

In [ ]:
yearly = movies.drop_duplicates('title').groupby('release_year').size()
yearly = yearly[(yearly.index >= 2000) & (yearly.index <= 2025)]

ax = yearly.plot(kind='line', marker='o', color='#e50914', linewidth=2)
ax.set_title('Titles released per year (2000–2025)')
ax.set_xlabel('Year'); ax.set_ylabel('Titles')
plt.tight_layout(); plt.show()

## 2. ROI by genre — and why we filter unreliable budgets first

Run this twice: once on the raw data, once after the $50K floor `core/etl.py` applies, to see why the filter matters.

In [ ]:
raw = pd.read_csv(etl.MOVIE_MARKET_CSV)
raw['roi_naive'] = np.where(raw['budget'] > 0, (raw['revenue'] - raw['budget']) / raw['budget'] * 100, np.nan)

exploded_raw = raw.assign(genre=raw['genres'].fillna('').str.split(',')).explode('genre')
exploded_raw['genre'] = exploded_raw['genre'].str.strip()
naive_roi_by_genre = exploded_raw.groupby('genre')['roi_naive'].mean().sort_values(ascending=False).head(10)
print('Naive ROI by genre (no budget floor) — note the absurd magnitudes:')
print(naive_roi_by_genre.round(0))

In [ ]:
exploded_clean = movies.dropna(subset=['roi']).assign(genre=movies['genre_list']).explode('genre')
exploded_clean['genre'] = exploded_clean['genre'].str.strip()
clean_roi_by_genre = exploded_clean.groupby('genre')['roi'].agg(['mean', 'count'])
clean_roi_by_genre = clean_roi_by_genre[clean_roi_by_genre['count'] >= 15].sort_values('mean', ascending=False).head(10)
print('Cleaned ROI by genre ($50K minimum-budget floor applied) — sane magnitudes:')
print(clean_roi_by_genre.round(1))

In [ ]:
ax = clean_roi_by_genre['mean'].clip(-200, 600).sort_values().plot(kind='barh', color='#46d369')
ax.set_title('Average ROI by genre (cleaned, top 10)')
ax.set_xlabel('ROI %')
plt.tight_layout(); plt.show()

## 3. Budget vs. revenue correlation

In [ ]:
financed = movies[movies['has_financials'] & (movies['revenue'] > 0)].drop_duplicates('title')
r = financed['budget'].corr(financed['revenue'])
print(f'Pearson r (budget, revenue) = {r:.3f}  ·  n = {len(financed):,}')

sample = financed.sample(min(800, len(financed)), random_state=42)
fig, ax = plt.subplots()
ax.scatter(sample['budget']/1e6, sample['revenue']/1e6, alpha=0.4, s=20, color='#3b9ef8')
ax.set_xlabel('Budget ($M)'); ax.set_ylabel('Revenue ($M)')
ax.set_title(f'Budget vs Revenue (r = {r:.2f})')
plt.tight_layout(); plt.show()

## 4. Real Netflix engagement — most-watched genres

Remember: `All-Time` rows are excluded from sums in the dashboard to avoid double-counting against the per-half-year rows (see `data/DATA_DICTIONARY.md`). Same rule applied here.

In [ ]:
period_only = engagement[engagement['report_period'] != 'All-Time']
by_genre = period_only.groupby('primary_genre')['views_millions'].sum().sort_values(ascending=False).head(10)

ax = by_genre.sort_values().plot(kind='barh', color='#f5c518')
ax.set_title('Total tracked views by genre, millions (All-Time rows excluded)')
ax.set_xlabel('Views (millions)')
plt.tight_layout(); plt.show()

In [ ]:
trend = period_only.groupby('report_period')[['views_millions', 'hours_viewed_millions']].sum()
order = etl.ENGAGEMENT_PERIOD_ORDER
trend = trend.reindex(order)
print(trend)

ax = trend['views_millions'].plot(kind='bar', color='#a855f7')
ax.set_title('Tracked views by report period')
ax.set_ylabel('Views (millions)')
plt.tight_layout(); plt.show()

## 5. Next steps

Ideas for extending this notebook:
- Pull in `core.forecasting` and plot the confidence band manually with matplotlib instead of Chart.js.
- Run the ANOVA from `core.analytics.get_genre_roi_significance` for individual countries, not just the global market.
- Cluster genres by their ROI *and* rating profile (k-means on the two standardized columns) to find a "quality vs. profitability" map.